# 04 — HQ200 car-only reference mesh cleaning

The supplied scanner mesh contains the car and background geometry. This notebook lets you inspect every scene,
choose an explicit 3D crop, preview it, and approve it before export. Raw OBJ files are never modified.

Output: `data_processed/reference_meshes/3DRealCar/<scene>/car_reference.obj`.
Because automatic foreground extraction from a connected room/car mesh is unreliable, every exported crop requires
manual approval. The saved JSON makes the process reproducible.


In [ ]:
%pip -q install trimesh scipy pandas matplotlib ipywidgets


In [ ]:
import json, shutil, subprocess, sys
from pathlib import Path
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, clear_output
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
PROJECT_ROOT = Path("/content/drive/MyDrive/ITU/3D/Thesis")
CODE_ROOT = Path("/content/Project_Thesis_code")
REPOSITORY = "ht" + "tps:" + chr(47)*2 + "github.com" + chr(47) + "katlit" + chr(47) + "Project_Thesis.git"
BRANCH = "codex/hq200-example-notebook"
if CODE_ROOT.exists() and not (CODE_ROOT / ".git").is_dir(): shutil.rmtree(CODE_ROOT)
command = (["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY, str(CODE_ROOT)]
           if not CODE_ROOT.exists() else ["git", "-C", str(CODE_ROOT), "pull", "--ff-only", "origin", BRANCH])
subprocess.run(command, check=True)
sys.path.insert(0, str(CODE_ROOT / "code"))

from src.mesh_cleanup import crop_mesh, export_clean_reference, load_triangle_mesh, mesh_summary, sample_for_display


In [ ]:
def find_unique_dir(names, roots):
    matches = [root / name for root in roots for name in names if (root / name).is_dir()]
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one HQ200 root, found: {matches}")
    return matches[0]

HQ200_ROOT = find_unique_dir(["3DrealCarHQ200", "HQ200"], [PROJECT_ROOT / "data", PROJECT_ROOT])
if (HQ200_ROOT / "3DrealCarHQ200").is_dir(): HQ200_ROOT /= "3DrealCarHQ200"
OUTPUT_ROOT = PROJECT_ROOT / "data_processed/reference_meshes/3DRealCar"
CONFIG_PATH = PROJECT_ROOT / "splits/reference_mesh_crop_boxes.json"
mesh_paths = {path.parent.name: path for path in HQ200_ROOT.glob("*/textured_output.obj")}
if not mesh_paths: raise FileNotFoundError(f"No textured_output.obj files below {HQ200_ROOT}")
scenes = sorted(mesh_paths)
rows = [mesh_summary(load_triangle_mesh(path), scene, path) for scene, path in mesh_paths.items()]
display(pd.DataFrame(rows).sort_values("scene").round(3))
print("Scenes:", len(scenes), "| raw meshes are read-only")


## Interactive crop and approval

The sliders are normalized to each mesh's complete bounds: `0` is its minimum and `1` its maximum on that axis.
Adjust the three ranges until the right-hand preview contains the complete car but no surrounding scan. Check at
least two rotations mentally by rerunning the preview; the plot shows two automatically. Click **Approve and save**
only when the complete car is retained. Saving the crop does not yet export a mesh.


In [ ]:
saved = json.loads(CONFIG_PATH.read_text()) if CONFIG_PATH.is_file() else {}
default_box = [[0.25, 0.75], [0.05, 0.95], [0.25, 0.75]]
scene_widget = widgets.Dropdown(options=scenes, description="Scene:", layout=widgets.Layout(width="750px"))
sliders = [widgets.FloatRangeSlider(value=default_box[i], min=0, max=1, step=.01,
           description=axis, continuous_update=False, layout=widgets.Layout(width="700px"))
           for i, axis in enumerate(["X", "Y", "Z"])]
preview_button = widgets.Button(description="Preview crop", button_style="info")
approve_button = widgets.Button(description="Approve and save", button_style="success")
output = widgets.Output()

def current_box(): return [list(slider.value) for slider in sliders]

def load_saved_box(change=None):
    box = saved.get(scene_widget.value, {}).get("normalized_box", default_box)
    for slider, limits in zip(sliders, box): slider.value = tuple(limits)

def show_preview(_=None):
    with output:
        clear_output(wait=True)
        scene = scene_widget.value
        original = load_triangle_mesh(mesh_paths[scene])
        try: cropped = crop_mesh(original, current_box())
        except Exception as error:
            print("Invalid crop:", error); return
        original_points = sample_for_display(original, 15_000)
        cropped_points = sample_for_display(cropped, 20_000)
        fig = plt.figure(figsize=(18, 8))
        for index, (points, title, azimuth) in enumerate([
            (original_points, "Original mesh including background", -65),
            (cropped_points, "Proposed car-only crop", -65),
            (cropped_points, "Proposed crop — second angle", 25),
        ], start=1):
            axis = fig.add_subplot(1, 3, index, projection="3d")
            axis.scatter(*points.T, s=.15, c="0.25")
            axis.set_title(title); axis.set_box_aspect(np.ptp(points, axis=0).clip(min=1e-6))
            axis.view_init(18, azimuth)
        plt.tight_layout(); plt.show()
        display(pd.DataFrame([mesh_summary(cropped, scene)]).round(3))

def approve(_):
    scene = scene_widget.value
    original = load_triangle_mesh(mesh_paths[scene])
    cropped = crop_mesh(original, current_box())
    saved[scene] = {"normalized_box": current_box(), "approved": True,
                    "raw_mesh": str(mesh_paths[scene]), "cropped_summary": mesh_summary(cropped, scene)}
    CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
    CONFIG_PATH.write_text(json.dumps(saved, indent=2))
    with output: print("Approved and saved:", scene, "→", CONFIG_PATH)

scene_widget.observe(load_saved_box, names="value")
preview_button.on_click(show_preview); approve_button.on_click(approve)
load_saved_box()
display(widgets.VBox([scene_widget, *sliders, widgets.HBox([preview_button, approve_button]), output]))
show_preview()


## Approval status and guarded batch export


In [ ]:
saved = json.loads(CONFIG_PATH.read_text()) if CONFIG_PATH.is_file() else {}
status = pd.DataFrame([{"scene": scene, "approved": bool(saved.get(scene, {}).get("approved", False)),
                        "already_exported": (OUTPUT_ROOT / scene / "car_reference.obj").is_file()}
                       for scene in scenes])
display(status)
print("Approved:", int(status.approved.sum()), "/", len(status))


In [ ]:
RUN_EXPORT = False
OVERWRITE = False

if RUN_EXPORT:
    unapproved = [scene for scene in scenes if not saved.get(scene, {}).get("approved", False)]
    if unapproved:
        raise RuntimeError(f"Approve every scene before batch export. Missing: {unapproved}")
    export_rows = []
    for scene in scenes:
        original = load_triangle_mesh(mesh_paths[scene])
        cleaned = crop_mesh(original, saved[scene]["normalized_box"])
        destination = OUTPUT_ROOT / scene / "car_reference.obj"
        export_clean_reference(cleaned, destination, overwrite=OVERWRITE)
        export_rows.append(mesh_summary(cleaned, scene, destination))
    manifest = pd.DataFrame(export_rows)
    manifest.to_csv(OUTPUT_ROOT / "manifest.csv", index=False)
    display(manifest.round(3)); print("Saved:", OUTPUT_ROOT)
else:
    print("Dry run. After approving every scene, set RUN_EXPORT=True.")
